# 4. Optimal execution (Almgren-Chriss)

**Level:** L2. **Tags:** `execution`, `almgren-chriss`, `market-impact`.
**Library:** `eq_algo.almgren_chriss` (`python/equity/08-algo-execution`).

**Plain idea.** Selling fast costs impact; selling slowly leaves you exposed
to price moves. Almgren and Chriss trade the two off with a risk-aversion
parameter $\lambda$: $\lambda = 0$ gives a straight line (TWAP), larger
$\lambda$ front-loads.

**Formal statement** (Almgren and Chriss, "Optimal execution of portfolio
transactions", *Journal of Risk* 3, 2000): minimise $E + \lambda V$ with
$E = \tfrac12\gamma X^2 + \tfrac{\tilde\eta}{\tau}\sum n_k^2$ and
$V = \sigma^2\tau\sum x_k^2$. The optimum is
$x_j = X\sinh(\kappa(T - t_j))/\sinh(\kappa T)$.

In [ ]:
import _paths  # noqa: F401  (puts the library on sys.path)

In [ ]:
import math  # noqa: F401

import numpy as np
from eq_algo.almgren_chriss import ACParams, ac_cost_moments, ac_trajectory  # noqa: F401

## Task 1: trajectories for three risk aversions

Sell 1,000,000 shares over one day in 20 slices with $\sigma = 0.5$ per
share per day and $\eta = 2.5\times10^{-6}$, $\gamma = 2.5\times10^{-7}$.
Store the three trajectories for $\lambda \in \{0, 10^{-6}, 10^{-5}\}$ in
`traj` (a dict).

In [ ]:
p = ACParams(total_shares=1e6, n_slices=20, total_time=1.0, sigma=0.5, eta=2.5e-6, gamma=2.5e-7)
traj = ...
# Exercise: replace the ... above, then run the check cell below.

**Independent check.** The objective is a quadratic in the interior
holdings $x_1..x_{N-1}$, so the optimum solves a tridiagonal linear system:
$\frac{2\tilde\eta}{\tau}(2x_k - x_{k-1} - x_{k+1}) + 2\lambda\sigma^2\tau x_k = 0$.
We solve it with `numpy.linalg.solve`, which shares no code with the
library's sinh formula.

In [ ]:
def brute_force(p, lam):
    n, tau, et = p.n_slices, p.tau, p.eta_tilde
    a = np.zeros((n - 1, n - 1))
    b = np.zeros(n - 1)
    for k in range(n - 1):
        a[k, k] = 2 * et / tau * 2 + 2 * lam * p.sigma**2 * tau
        if k > 0:
            a[k, k - 1] = -2 * et / tau
        if k < n - 2:
            a[k, k + 1] = -2 * et / tau
    b[0] = 2 * et / tau * p.total_shares
    return np.concatenate([[p.total_shares], np.linalg.solve(a, b), [0.0]])

for lam, x in traj.items():
    assert np.max(np.abs(x - brute_force(p, lam))) < 1e-6 * p.total_shares
assert np.allclose(traj[0.0], np.linspace(1e6, 0, 21))

## Task 2: the efficient frontier is convex and ordered

For a grid of $\lambda$ compute (sd, E). Higher $\lambda$ must give lower
variance and higher expected cost.

In [ ]:
lams = np.geomspace(1e-8, 1e-4, 9)
frontier = ...  # rows of (E, V)
# Exercise: replace the ... above, then run the check cell below.

In [ ]:
assert np.all(np.diff(frontier[:, 0]) > 0)
assert np.all(np.diff(frontier[:, 1]) < 0)
# TWAP has the lowest possible expected cost among all schedules.
assert ac_cost_moments(p, traj[0.0])[0] <= frontier[0, 0] + 1e-6

**Takeaways.** A closed form is trusted once a dumb brute-force solve agrees
with it, and the frontier has the shape the theory says.